# 🎙️ VoiceBatch Studio v2.0.8 - [Final Fixed]
इसमें Anti-Sleep Mode और Punctuation Fix शामिल है।

In [ ]:
# @title 💤 Step 0: Anti-Sleep Mode (कोलाब को जागृत रखने के लिए)
import IPython
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
  console.log("Clicking on connect button");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect,60000)
'''))
print("🚀 Anti-Sleep सक्रिय है! अब कोलाब बंद नहीं होगा।")

In [ ]:
# @title 📥 Step 1: GitHub Root & Engine Setup
import os
GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "VoiceBatch_Studio" # @param {type:"string"}

os.makedirs(f"{REPO_NAME}/configs", exist_ok=True)
os.makedirs(f"{REPO_NAME}/outputs", exist_ok=True)

print("⏳ जरूरी लाइब्रेरी इंस्टॉल हो रही हैं...")
!pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
print("✅ सिस्टम तैयार है!")

In [ ]:
# @title 🚀 Step 2: app.py (No-Stutter & Symbol Logic)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os, re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def fix_stutter(text):
    # सिंबल्स पर हकलाहट रोकने के लिए क्लीनिंग
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    return text

def voice_engine(text, audio_sample, speed, pitch, lang):
    if not audio_sample: return None
    
    cleaned_text = fix_stutter(text)
    temp_out = 'VoiceBatch_Studio/outputs/raw.wav'
    
    # Strict Language Enforcement
    tts.tts_to_file(
        text=cleaned_text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=temp_out,
        split_sentences=True
    )
    
    y, sr = librosa.load(temp_out)
    y, _ = librosa.effects.trim(y, top_db=25)
    
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    final_path = 'VoiceBatch_Studio/outputs/final_studio.wav'
    sf.write(final_path, y, sr)
    return final_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.8')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script यहाँ डालें', lines=8)
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr', 'bn'], label='Language Lock', value='hi')
            with gr.Row():
                spd = gr.Slider(0.8, 1.2, 1.0, step=0.01, label="Speed")
                ptc = gr.Slider(-3, 3, 0, step=1, label="Pitch")
            btn = gr.Button('Generate Voice 🚀', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Output')

    btn.click(voice_engine, [txt, smp, spd, ptc, lng], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप तैयार है! अब रन करें।")
!python app.py